# 05 — T1 graph models (CGCNN), GPU, checkpoint-resume
**Project:** MAG2D-NC | **Phase:** F6 / Step 1 | **Protocol:** v1.2 (GNN search
amendment — pending user ratification, disclosed in paper)

Ablation axis A2: does a structural graph representation beat tabular
descriptors at N=164? Frozen expectation: a NO is publishable.

Design:
- CGCNN-style crystal graph network, pure PyTorch (no PyG/dgl), graphs built
  from `structures_164/*.extxyz` via ASE periodic neighbor lists.
- Same outer CV as baselines: StratifiedGroupKFold(5) x 3 repeats x 5 seeds.
- v1.2 GNN search: 8 fixed random configs per fold, group-aware inner
  validation split, early stopping; equal treatment for any future GNN.
- **Checkpoint-resume:** unit = (model, seed, rep, fold) -> 75 units. Each unit
  writes its metrics + test predictions to disk on completion; rerunning the
  notebook skips finished units. Power loss costs at most one unit.
- A5 cost logging: per-unit wall time + peak VRAM.

Optional final cell probes the official `alignn` package; failure is reported,
not silently substituted.

## CONFIG

In [ ]:
from pathlib import Path
from datetime import datetime
import glob

CONFIG = {
    "PROJECT_ROOT": Path.home() / "MAG2D-NC",
    "SEEDS": [0,1,2,3,4], "N_SPLITS": 5, "N_REPEATS": 3,
    "GNN_CONFIGS": 8, "MAX_EPOCHS": 150, "PATIENCE": 25,
    "CUTOFF": 8.0, "MAX_NBRS": 12, "N_GAUSS": 40,
    "PRIMARY": "f1_macro",
}
QUICK_SMOKE = False   # True: tiny budgets for plumbing check only -- NEVER report

CKPT = CONFIG["PROJECT_ROOT"]/"checkpoints"/"T1_gnn"
if QUICK_SMOKE:
    CONFIG.update({"SEEDS":[0], "N_REPEATS":1, "GNN_CONFIGS":2, "MAX_EPOCHS":8, "PATIENCE":3})
    print("*** SMOKE MODE -- results not reportable ***")
    CKPT = CONFIG["PROJECT_ROOT"]/"checkpoints"/"T1_gnn_SMOKE"

(CKPT/"preds").mkdir(parents=True, exist_ok=True)
CONFIG["PROGRESS_CSV"] = CKPT/"progress.csv"

if not QUICK_SMOKE:
    assert "SMOKE" not in str(CONFIG["PROGRESS_CSV"]), "Full run must not write to the SMOKE folder!"
print("checkpoint dir:", CKPT)

## Dependencies + device
torch pypi wheel bundles CUDA on Linux; RTX 4500 Ada must appear below.
CPU fallback works but is ~20x slower — flag it if you see it.

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [("torch","torch"), ("ase","ase"), ("pandas","pandas")]:
    try:
        importlib.import_module(mod); print(pkg, "OK")
    except (ImportError, OSError):
        subprocess.run([sys.executable,"-m","pip","install",pkg], check=True)
        print(pkg, "installed")

import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "|", torch.cuda.get_device_name(0) if DEV=="cuda" else "(CPU fallback!)")

## Load labels + build graphs (cached)
Graphs are built once from the 164 extxyz files and cached to disk; rebuilds
are skipped on resume. Periodic neighbors within 8 A, capped at 12/atom,
distances expanded in 40 Gaussians.

In [ ]:
import numpy as np, pandas as pd, torch
from ase.io import read as ase_read
from ase.neighborlist import neighbor_list

df = pd.read_csv(CONFIG["PROJECT_ROOT"]/"dataset"/"spiral_labels_v2.csv")
assert len(df)==164
y = (df["label2"]=="non_collinear").astype(int).to_numpy()
groups = df["group_id"].to_numpy()
label4 = df["label4"].to_numpy()

GCACHE = CKPT/"graphs.pt"
if GCACHE.exists():
    graphs = torch.load(GCACHE, weights_only=False)
    print("graphs loaded from cache:", len(graphs))
else:
    centers = np.linspace(0, CONFIG["CUTOFF"], CONFIG["N_GAUSS"]); width = centers[1]-centers[0]
    graphs = []
    for uid in df["uid"]:
        at = ase_read(CONFIG["PROJECT_ROOT"]/"dataset"/"structures_164"/f"{uid}.extxyz")
        i, j, d = neighbor_list("ijd", at, CONFIG["CUTOFF"])
        keep = []
        for a in range(len(at)):                       # cap neighbors per atom
            idx = np.where(i==a)[0]
            keep.extend(idx[np.argsort(d[idx])][:CONFIG["MAX_NBRS"]])
        keep = np.array(keep, int)
        i, j, d = i[keep], j[keep], d[keep]
        ef = np.exp(-((d[:,None]-centers[None,:])/width)**2)
        graphs.append({"z": torch.tensor(at.numbers, dtype=torch.long),
                       "ei": torch.tensor(np.stack([i,j]), dtype=torch.long),
                       "ef": torch.tensor(ef, dtype=torch.float32)})
    torch.save(graphs, GCACHE)
    print("graphs built + cached:", len(graphs))
nbr_counts = [g["ei"].shape[1]/len(g["z"]) for g in graphs]
print(f"avg neighbors/atom: {np.mean(nbr_counts):.1f}")

## CGCNN model (compact, faithful message passing)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CGConv(nn.Module):
    def __init__(self, node_d, edge_d):
        super().__init__()
        self.lin = nn.Linear(2*node_d+edge_d, 2*node_d)
        self.bn = nn.BatchNorm1d(node_d)
    def forward(self, x, ei, ef):
        src, dst = ei
        z = torch.cat([x[dst], x[src], ef], dim=1)
        gate, core = self.lin(z).chunk(2, dim=1)
        msg = torch.sigmoid(gate) * F.softplus(core)
        agg = torch.zeros_like(x).index_add_(0, dst, msg)
        return F.softplus(x + self.bn(agg))

class CGCNN(nn.Module):
    def __init__(self, node_d=64, edge_d=40, n_conv=3, fc_d=64, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(100, node_d)
        self.convs = nn.ModuleList([CGConv(node_d, edge_d) for _ in range(n_conv)])
        self.head = nn.Sequential(nn.Linear(node_d, fc_d), nn.Softplus(),
                                  nn.Dropout(dropout), nn.Linear(fc_d, 2))
    def forward(self, batch):
        out = []
        for g in batch:                    # graphs are tiny; loop is fine
            x = self.emb(g["z"].to(DEV))
            for c in self.convs:
                x = c(x, g["ei"].to(DEV), g["ef"].to(DEV))
            out.append(x.mean(dim=0))
        return self.head(torch.stack(out))
print("model class ready")

## Training machinery (v1.2 search) + checkpoint-resume runner
Unit = (model, seed, rep, fold). Completed units are read from progress.csv and
skipped. Each unit: sample 8 configs (seeded), split train into fit/val by
GroupShuffleSplit, early-stop on val F1-macro, refit best config on full train,
predict test, write metrics row + preds npz IMMEDIATELY.

In [ ]:
import time, itertools, json as _j
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score

SPACE = {"node_d":[32,64,128], "n_conv":[2,3,4], "fc_d":[32,64,128],
         "lr":[1e-3,3e-3,1e-2], "weight_decay":[0.0,1e-5,1e-4],
         "dropout":[0.0,0.1,0.3], "batch":[16,32]}

def sample_cfgs(rng, k):
    return [{kk: vv[rng.integers(len(vv))] for kk, vv in SPACE.items()} for _ in range(k)]

def train_one(cfg, idx_fit, idx_val, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = CGCNN(cfg["node_d"], CONFIG["N_GAUSS"], cfg["n_conv"], cfg["fc_d"], cfg["dropout"]).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    w = torch.tensor([len(idx_fit)/max((y[idx_fit]==0).sum(),1),
                      len(idx_fit)/max((y[idx_fit]==1).sum(),1)], dtype=torch.float32).to(DEV)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(CONFIG["MAX_EPOCHS"]):
        model.train()
        perm = np.random.permutation(idx_fit)
        for b in range(0, len(perm), cfg["batch"]):
            bi = perm[b:b+cfg["batch"]]
            logits = model([graphs[i] for i in bi])
            loss = F.cross_entropy(logits, torch.tensor(y[bi], dtype=torch.long).to(DEV), weight=w)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pv = model([graphs[i] for i in idx_val]).argmax(1).cpu().numpy()
        f1v = f1_score(y[idx_val], pv, average="macro")
        if f1v > best_f1 + 1e-4:
            best_f1, best_state, bad = f1v, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= CONFIG["PATIENCE"]: break
    model.load_state_dict(best_state)
    return model, best_f1

def done_units():
    if CONFIG["PROGRESS_CSV"].exists():
        p = pd.read_csv(CONFIG["PROGRESS_CSV"])
        return set(zip(p["model"], p["seed"], p["rep"], p["fold"]))
    return set()

def run_gnn(model_name="cgcnn"):
    finished = done_units()
    print(f"resume: {len(finished)} units already done")
    for seed in CONFIG["SEEDS"]:
        for rep in range(CONFIG["N_REPEATS"]):
            rs = 1000*seed + rep
            outer = StratifiedGroupKFold(CONFIG["N_SPLITS"], shuffle=True, random_state=rs)
            for k, (tr, te) in enumerate(outer.split(np.zeros(len(y)), y, groups)):
                key = (model_name, seed, rep, k)
                if key in finished:
                    continue
                t0 = time.time()
                if DEV=="cuda": torch.cuda.reset_peak_memory_stats()
                rng = np.random.default_rng(rs*10+k)
                gss = GroupShuffleSplit(1, test_size=0.25, random_state=rs*10+k)
                fit_i, val_i = next(gss.split(tr, y[tr], groups[tr]))
                idx_fit, idx_val = tr[fit_i], tr[val_i]
                best = (-1, None)
                for cfg in sample_cfgs(rng, CONFIG["GNN_CONFIGS"]):
                    _, f1v = train_one(cfg, idx_fit, idx_val, seed)
                    if f1v > best[0]: best = (f1v, cfg)
                model, _ = train_one(best[1], tr, idx_val, seed)   # refit on full train (val reused for early-stop only)
                model.eval()
                with torch.no_grad():
                    yp = model([graphs[i] for i in te]).argmax(1).cpu().numpy()
                row = {"model": model_name, "seed": seed, "rep": rep, "fold": k,
                       "f1_macro": f1_score(y[te], yp, average="macro"),
                       "mcc": matthews_corrcoef(y[te], yp),
                       "bal_acc": balanced_accuracy_score(y[te], yp),
                       "cfg": _j.dumps(best[1], default=str),
                       "sec": round(time.time()-t0,1),
                       "vram_mb": round(torch.cuda.max_memory_allocated()/1e6,1) if DEV=="cuda" else 0}
                pd.DataFrame([row]).to_csv(CONFIG["PROGRESS_CSV"], mode="a",
                                           header=not CONFIG["PROGRESS_CSV"].exists(), index=False)
                np.savez(CKPT/"preds"/f"{model_name}_s{seed}_r{rep}_f{k}.npz",
                         te=te, y_true=y[te], y_pred=yp)
                print(f"[{model_name}] s{seed} r{rep} f{k} f1={row['f1_macro']:.3f} "
                      f"{row['sec']}s {row['vram_mb']}MB", flush=True)
    print("all units complete")
print("runner ready")

## RUN (resume-safe — rerun this cell after any interruption)

In [ ]:
run_gnn("cgcnn")

## Summary vs baselines

In [ ]:
prog = pd.read_csv(CONFIG["PROGRESS_CSV"])
prog = prog[prog.model=="cgcnn"]
v = prog["f1_macro"].to_numpy()
rng_b = np.random.default_rng(0)
bs = rng_b.choice(v, size=(10000, len(v)), replace=True).mean(axis=1)
print(f"CGCNN: f1_macro {v.mean():.3f} ± {v.std(ddof=1):.3f} "
      f"[{np.percentile(bs,2.5):.3f}, {np.percentile(bs,97.5):.3f}] (n={len(v)})")
print(f"cost: median {prog['sec'].median():.0f}s/unit | peak VRAM {prog['vram_mb'].max():.0f} MB")
print("\nReference (nb03 full run): logreg 0.615 [0.590,0.639] | lgbm 0.608 [0.581,0.634]")
print("A2 comparison stats (Wilcoxon vs baselines) will be computed in the F6 wrap-up "
      "notebook against runs.csv — paste this summary back first.")

## Optional: official ALIGNN probe (reports, never silently substitutes)

In [ ]:
try:
    import importlib
    importlib.import_module("alignn")
    print("alignn IMPORTABLE — we can plan the official-ALIGNN arm next.")
except Exception as e:
    print("alignn NOT available:", repr(e))
    print("Decision returns to advisor: try pip install alignn (dgl dependency, "
          "may fail on WSL) or ratify a PyG-based angle-aware equivalent as the "
          "second graph model (protocol amendment).")

## ALIGNN-lite (protocol v1.3)
Official `alignn` dropped: mandatory dgl dependency incompatible with the
current torch ecosystem (dgl 2.1.0 / torchdata 0.11 datapipes conflict,
Aug 2026) -- disclosed in the paper. This variant keeps ALIGNN's defining idea:
bond-angle information processed on the line graph, alternating with atom-graph
convolutions. Same v1.2 search scheme (8 configs), same checkpoint-resume,
same progress.csv (model="alignn_lite").

In [ ]:
# Line-graph cache: edges WITH displacement vectors -> triplet angle features
import numpy as np, torch
from ase.io import read as ase_read
from ase.neighborlist import neighbor_list

N_ANGLE = 20
LGCACHE = CKPT/"graphs_lg.pt"
if LGCACHE.exists():
    graphs_lg = torch.load(LGCACHE, weights_only=False)
    print("line graphs loaded:", len(graphs_lg))
else:
    dcent = np.linspace(0, CONFIG["CUTOFF"], CONFIG["N_GAUSS"]); dw = dcent[1]-dcent[0]
    acent = np.linspace(-1, 1, N_ANGLE); aw = acent[1]-acent[0]
    graphs_lg = []
    for uid in df["uid"]:
        at = ase_read(CONFIG["PROJECT_ROOT"]/"dataset"/"structures_164"/f"{uid}.extxyz")
        i, j, d, D = neighbor_list("ijdD", at, CONFIG["CUTOFF"])
        keep = []
        for a in range(len(at)):
            idx = np.where(i == a)[0]
            keep.extend(idx[np.argsort(d[idx])][:CONFIG["MAX_NBRS"]])
        keep = np.array(keep, int)
        i, j, d, D = i[keep], j[keep], d[keep], D[keep]
        ef = np.exp(-((d[:,None]-dcent[None,:])/dw)**2)
        # triplets: edge pairs sharing the SOURCE atom -> angle between bond vectors
        t1, t2, ang = [], [], []
        for a in range(len(at)):
            idx = np.where(i == a)[0]
            for p in range(len(idx)):
                for q in range(len(idx)):
                    if p == q: continue
                    v1, v2 = D[idx[p]], D[idx[q]]
                    c = float(np.dot(v1, v2)/(np.linalg.norm(v1)*np.linalg.norm(v2)+1e-9))
                    t1.append(idx[p]); t2.append(idx[q]); ang.append(c)
        af = np.exp(-((np.clip(ang,-1,1)[:,None]-acent[None,:])/aw)**2)
        graphs_lg.append({"z": torch.tensor(at.numbers, dtype=torch.long),
                          "ei": torch.tensor(np.stack([i,j]), dtype=torch.long),
                          "ef": torch.tensor(ef, dtype=torch.float32),
                          "t1": torch.tensor(np.array(t1), dtype=torch.long),
                          "t2": torch.tensor(np.array(t2), dtype=torch.long),
                          "af": torch.tensor(af, dtype=torch.float32)})
    torch.save(graphs_lg, LGCACHE)
    print("line graphs built + cached:", len(graphs_lg))
print("avg triplets/structure:", int(np.mean([g["t1"].shape[0] for g in graphs_lg])))

In [ ]:
import torch.nn as nn, torch.nn.functional as F

class GatedUpdate(nn.Module):
    """Edge-gated aggregation: target features updated from (source, context) pairs."""
    def __init__(self, dim, ctx):
        super().__init__()
        self.lin = nn.Linear(2*dim+ctx, 2*dim); self.bn = nn.BatchNorm1d(dim)
    def forward(self, x, src_idx, dst_idx, ctx):
        z = torch.cat([x[dst_idx], x[src_idx], ctx], dim=1)
        gate, core = self.lin(z).chunk(2, dim=1)
        msg = torch.sigmoid(gate)*F.softplus(core)
        agg = torch.zeros_like(x).index_add_(0, dst_idx, msg)
        return F.softplus(x + self.bn(agg))

class ALIGNNLite(nn.Module):
    def __init__(self, node_d=64, edge_d=64, n_layer=3, fc_d=64, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(100, node_d)
        self.eproj = nn.Linear(CONFIG["N_GAUSS"], edge_d)
        self.edge_upd = nn.ModuleList([GatedUpdate(edge_d, N_ANGLE) for _ in range(n_layer)])
        self.node_upd = nn.ModuleList([GatedUpdate(node_d, edge_d) for _ in range(n_layer)])
        self.head = nn.Sequential(nn.Linear(node_d, fc_d), nn.Softplus(),
                                  nn.Dropout(dropout), nn.Linear(fc_d, 2))
    def forward(self, batch):
        out = []
        for g in batch:
            x = self.emb(g["z"].to(DEV))
            e = F.softplus(self.eproj(g["ef"].to(DEV)))
            src, dst = g["ei"].to(DEV)
            t1, t2, af = g["t1"].to(DEV), g["t2"].to(DEV), g["af"].to(DEV)
            for eu, nu in zip(self.edge_upd, self.node_upd):
                e = eu(e, t2, t1, af)        # line graph: angle-aware edge update
                x = nu(x, src, dst, e)       # atom graph: node update w/ edges
            out.append(x.mean(dim=0))
        return self.head(torch.stack(out))
print("ALIGNNLite ready")

In [ ]:
# v1.2 search space for alignn_lite (equal treatment) + generic unit runner
SPACE_LITE = {"node_d":[32,64,128], "edge_d":[32,64], "n_layer":[2,3,4],
              "fc_d":[32,64,128], "lr":[1e-3,3e-3,1e-2],
              "weight_decay":[0.0,1e-5,1e-4], "dropout":[0.0,0.1,0.3], "batch":[16,32]}

def train_one_lite(cfg, idx_fit, idx_val, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = ALIGNNLite(cfg["node_d"], cfg["edge_d"], cfg["n_layer"],
                       cfg["fc_d"], cfg["dropout"]).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    w = torch.tensor([len(idx_fit)/max((y[idx_fit]==0).sum(),1),
                      len(idx_fit)/max((y[idx_fit]==1).sum(),1)], dtype=torch.float32).to(DEV)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(CONFIG["MAX_EPOCHS"]):
        model.train()
        perm = np.random.permutation(idx_fit)
        for b in range(0, len(perm), cfg["batch"]):
            bi = perm[b:b+cfg["batch"]]
            logits = model([graphs_lg[i] for i in bi])
            loss = F.cross_entropy(logits, torch.tensor(y[bi], dtype=torch.long).to(DEV), weight=w)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pv = model([graphs_lg[i] for i in idx_val]).argmax(1).cpu().numpy()
        f1v = f1_score(y[idx_val], pv, average="macro")
        if f1v > best_f1 + 1e-4:
            best_f1, best_state, bad = f1v, {k:v.detach().clone() for k,v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= CONFIG["PATIENCE"]: break
    model.load_state_dict(best_state)
    return model, best_f1

def sample_cfgs_lite(rng, k):
    return [{kk: vv[rng.integers(len(vv))] for kk, vv in SPACE_LITE.items()} for _ in range(k)]

def run_lite():
    finished = done_units()
    print(f"resume: {sum(1 for u in finished if u[0]=='alignn_lite')} lite units done")
    for seed in CONFIG["SEEDS"]:
        for rep in range(CONFIG["N_REPEATS"]):
            rs = 1000*seed + rep
            outer = StratifiedGroupKFold(CONFIG["N_SPLITS"], shuffle=True, random_state=rs)
            for k, (tr, te) in enumerate(outer.split(np.zeros(len(y)), y, groups)):
                if ("alignn_lite", seed, rep, k) in finished: continue
                t0 = time.time()
                if DEV=="cuda": torch.cuda.reset_peak_memory_stats()
                rng = np.random.default_rng(rs*10+k)
                gss = GroupShuffleSplit(1, test_size=0.25, random_state=rs*10+k)
                fit_i, val_i = next(gss.split(tr, y[tr], groups[tr]))
                idx_fit, idx_val = tr[fit_i], tr[val_i]
                best = (-1, None)
                for cfg in sample_cfgs_lite(rng, CONFIG["GNN_CONFIGS"]):
                    _, f1v = train_one_lite(cfg, idx_fit, idx_val, seed)
                    if f1v > best[0]: best = (f1v, cfg)
                model, _ = train_one_lite(best[1], tr, idx_val, seed)
                model.eval()
                with torch.no_grad():
                    yp = model([graphs_lg[i] for i in te]).argmax(1).cpu().numpy()
                row = {"model":"alignn_lite","seed":seed,"rep":rep,"fold":k,
                       "f1_macro": f1_score(y[te], yp, average="macro"),
                       "mcc": matthews_corrcoef(y[te], yp),
                       "bal_acc": balanced_accuracy_score(y[te], yp),
                       "cfg": _j.dumps(best[1], default=str),
                       "sec": round(time.time()-t0,1),
                       "vram_mb": round(torch.cuda.max_memory_allocated()/1e6,1) if DEV=="cuda" else 0}
                pd.DataFrame([row]).to_csv(CONFIG["PROGRESS_CSV"], mode="a",
                                           header=not CONFIG["PROGRESS_CSV"].exists(), index=False)
                np.savez(CKPT/"preds"/f"alignn_lite_s{seed}_r{rep}_f{k}.npz",
                         te=te, y_true=y[te], y_pred=yp)
                print(f"[lite] s{seed} r{rep} f{k} f1={row['f1_macro']:.3f} "
                      f"{row['sec']}s {row['vram_mb']}MB", flush=True)
    print("alignn_lite complete")

run_lite()